# RSNA STR 肺栓塞（PED）本地切 patch → npy

数据根目录（解压后的竞赛包，含 `train/`、`test/`、`train.csv` 等）：**`D:\111\rsna-str-pulmonary-embolism-detection`**。

典型结构：`train_part1/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm`（叶子目录仅含 `.dcm`，无子目录）。本笔记本默认**只对 `train_part1` 切 patch**，不扫描整个 `train/`。

**切 patch 逻辑**与 `colab_rsna2022_csfd_preprocess.ipynb` / `preprocess/proc_spatial_sequence.py` 一致：

- 每个 3D 序列：轴向 **z≥128** 时 **50** 个 npy，否则 **33** 个（`int(50/1.5)`）
- patch **128³**，`Resize` + `ScaleIntensityRangePercentiles(1, 99.9)`
- 输出 **扁平** 保存在 **`NPY_ROOT`**（默认 `D:\111\npy`），命名前缀 **`RSNA-PED_`**，体积 ID 为相对数据根的 `train_...` / `test_...` 路径（避免 Study/Series 重名）。

可选 **`PED_INCLUDE_TEST = False`**：不处理 `test/`（仅预训练常用 train）。

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nibabel", "monai", "SimpleITK", "tqdm"])

0

In [2]:
import os

# 解压后的 RSNA STR PED 根目录（其下应有 train、test 等）
DATA_ROOT = r"D:\111\rsna-str-pulmonary-embolism-detection"

# 切 patch 输出目录（扁平）
NPY_ROOT = r"D:\111\npy"
os.makedirs(NPY_ROOT, exist_ok=True)

# 仅处理 DATA_ROOT 下该训练子目录（相对路径），例如 train_part1 → D:\111\...\train_part1
PED_TRAIN_REL = "train_part1"

# 是否也处理 test/（默认 False）
PED_INCLUDE_TEST = True

先运行上面两格，再运行下一格（完整实现），最后运行 **`process_ped_train()`**（仅 `DATA_ROOT/PED_TRAIN_REL`，默认 `train_part1`）；若 `PED_INCLUDE_TEST=True` 再运行 **`process_ped_test()`**。

In [3]:
# RSNA-PED：切 patch 共用
import os
import random
import numpy as np
import nibabel as nib
import SimpleITK as sitk
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

DS_PREFIX = "RSNA-PED"

PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode="trilinear"),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume_zyx(image_zyx, patch_size_list, patch_num, save_root, start_index, tar_img_size, ds_name, vol_id):
    if len(image_zyx.shape) == 4:
        return []
    image = image_zyx
    n, patch_path_list = 0, []
    image_index = 0
    _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
    for i in range(_patch_num):
        n += 1
        patch_size = random.choice(patch_size_list)
        image_patch, _ = cut_patch(image, patch_size)
        image_patch = image_patch.transpose((2, 1, 0))
        image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
        save_name = os.path.join(save_root, "%s_%s_%s_%d.nii.gz" % (ds_name, vol_id, image_index, start_index + n))
        patch_path_list.append(save_name)
        np.save(save_name.replace(".nii.gz", ".npy"), image_patch)
    return patch_path_list


def cut_and_save_one_nii(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size, vol_id=None, ds_name=None):
    image, _affine = load_nii_data(image_file)
    path_parts = image_file.replace("\\", "/").split("/")
    if vol_id is None:
        ds_name = ds_name or DS_PREFIX
        nii_id = path_parts[-1].split(".nii.gz")[0].split(".nii")[0].split(".mha")[0]
    else:
        nii_id = vol_id
        ds_name = ds_name or DS_PREFIX
    if len(image.shape) == 4:
        return []
    image = image.transpose((2, 1, 0))
    return cut_and_save_one_volume_zyx(image, patch_size_list, patch_num, save_root, start_index, tar_img_size, ds_name, nii_id)


def load_dicom_series_zyx(series_dir):
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(series_dir)
    if not names:
        return None
    reader.SetFileNames(names)
    itk = reader.Execute()
    arr = sitk.GetArrayFromImage(itk)
    return arr.astype(np.float32)


def collect_nii_files(root):
    out = []
    if not os.path.isdir(root):
        return out
    for dirpath, _, files in os.walk(root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm:
            continue
        if "segmentations" in norm.split("/"):
            continue
        for f in files:
            if f.endswith(".nii.gz") or f.endswith(".nii"):
                out.append(os.path.join(dirpath, f))
    return sorted(out)


def collect_dicom_series_dirs(root):
    series_dirs = []
    if not os.path.isdir(root):
        return series_dirs
    for dirpath, dirnames, filenames in os.walk(root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm:
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if not dcms:
            continue
        if dirnames:
            continue
        series_dirs.append(dirpath)
    return sorted(series_dirs)


def _vol_id_from_series(series_dir, data_root):
    rel = os.path.relpath(series_dir, data_root).replace("\\", "/")
    return rel.replace("/", "_")


def process_ped_split(split_name, label=""):
    """split_name: DATA_ROOT 下的子目录名，如 train_part1、train、test 等。"""
    SCAN_ROOT = os.path.join(DATA_ROOT, split_name)
    if not os.path.isdir(SCAN_ROOT):
        print(f"[{label}] 跳过：目录不存在 -> {SCAN_ROOT}")
        return 0
    os.makedirs(NPY_ROOT, exist_ok=True)
    nii_list = collect_nii_files(SCAN_ROOT)
    dicom_list = collect_dicom_series_dirs(SCAN_ROOT)
    print(f"\n========== [{label}] ==========")
    print(f"SCAN_ROOT = {SCAN_ROOT}")
    print(f"NIfTI: {len(nii_list)} | DICOM 序列: {len(dicom_list)}")
    if len(nii_list) == 0 and len(dicom_list) == 0:
        print("未找到 .nii 或 DICOM 叶子序列。")
        return 0
    patch_list_all = []
    for path in tqdm(nii_list, desc=f"{label} NIfTI"):
        rel = os.path.relpath(path, DATA_ROOT).replace("\\", "/")
        vol_id = rel.replace("/", "_").replace(".nii.gz", "").replace(".nii", "")
        pl = cut_and_save_one_nii(
            path, PATCH_SIZE_LIST, PATCH_NUM, NPY_ROOT, START_INDEX, TAR_IMG_SIZE,
            vol_id=vol_id, ds_name=DS_PREFIX,
        )
        patch_list_all += pl
    for series_dir in tqdm(dicom_list, desc=f"{label} DICOM"):
        try:
            vol = load_dicom_series_zyx(series_dir)
        except Exception as e:
            print(f"跳过: {series_dir}\n  {e}")
            continue
        if vol is None:
            continue
        vol_id = _vol_id_from_series(series_dir, DATA_ROOT)
        pl = cut_and_save_one_volume_zyx(
            vol, PATCH_SIZE_LIST, PATCH_NUM, NPY_ROOT, START_INDEX, TAR_IMG_SIZE, DS_PREFIX, vol_id,
        )
        patch_list_all += pl
    print(f"[{label}] 本块写入 npy 数: {len(patch_list_all)} -> {NPY_ROOT}")
    return len(patch_list_all)


def process_ped_train():
    return process_ped_split(PED_TRAIN_REL, PED_TRAIN_REL)


def process_ped_test():
    return process_ped_split("test", "test")


print("已就绪：process_ped_train()；可选 process_ped_test()（需 PED_INCLUDE_TEST=True 时再跑）")


c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


已就绪：process_ped_train()；可选 process_ped_test()（需 PED_INCLUDE_TEST=True 时再跑）


In [4]:
# 仅处理 train_part1（见配置格 PED_TRAIN_REL）                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  555555555555555555555555555555555b
process_ped_train()


========== [train_part1] ==========
SCAN_ROOT = D:\111\rsna-str-pulmonary-embolism-detection\train_part1
NIfTI: 0 | DICOM 序列: 2440


train_part1 NIfTI: 0it [00:00, ?it/s]
train_part1 DICOM: 100%|██████████| 2440/2440 [1:26:41<00:00,  2.13s/it]

[train_part1] 本块写入 npy 数: 57576 -> D:\111\npy


57576

In [7]:
# 可选：处理 test/（将上一配置格中 PED_INCLUDE_TEST 设为 True 后运行本格）
if PED_INCLUDE_TEST:
    process_ped_test()
else:
    print("PED_INCLUDE_TEST=False，已跳过 test。改为 True 后重跑配置格再执行本格。")


========== [test] ==========
SCAN_ROOT = D:\111\rsna-str-pulmonary-embolism-detection\test
NIfTI: 0 | DICOM 序列: 650


test NIfTI: 0it [00:00, ?it/s]
test DICOM: 100%|██████████| 650/650 [18:28<00:00,  1.71s/it]

[test] 本块写入 npy 数: 11564 -> D:\111\npy


In [6]:
import shutil
from pathlib import Path

src = Path(r"D:\111\npy")
dst = Path(r"D:\111\npy8")
dst.mkdir(parents=True, exist_ok=True)

if not src.is_dir():
    raise FileNotFoundError(f"不存在或不是目录: {src.resolve()}")

files = sorted(p for p in src.iterdir() if p.is_file())
k = len(files) // 2

for p in files[:k]:
    shutil.move(str(p), dst / p.name)

print(f"共移动 {k} / {len(files)} 个文件 -> {dst}")

共移动 19192 / 38384 个文件 -> D:\111\npy8
